In [2]:
import boto3
import csv
from datetime import datetime, timedelta, timezone
import json
import time # Rate limit 시 대기를 위해
import pandas as pd

# --- 설정 ---
REGION_NAME = "us-west-2"  # AWS 리전

REGION_NAMES = ["us-west-2", "eu-west-1", "us-east-1", "ap-northeast-1"]
CLUSTER_NAMES = ["kubecaps-d2", "kubecaps-e1", "kubecaps-u1", "kubecaps-a1"]
LOOKBACK_DAYS = 1  # 최근 N일간의 이벤트를 조회
OUTPUT_CSV_FILE = f'ec2_instance_details_{REGION_NAME}.csv' # 출력 CSV 파일명

# EC2 인스턴스 시작과 관련된 주요 이벤트
# RunInstances: 새 인스턴스 시작 시 발생. 인스턴스 ID, 타입, 시작 시점 태그 등 상세 정보 포함.
EVENT_NAME_FOR_INSTANCE_CREATION = 'RunInstances'
# CreateTags: 리소스에 태그를 생성/업데이트할 때 발생.
# 이번 요청은 "EC2 시작 이벤트와 태그 생성 클라우드트레일 이벤트 긁어오기" 후
# "인스턴스ID, 인스턴스 타입, 태그, 시작시간"을 CSV로 정리하는 것입니다.
# 따라서, RunInstances 이벤트를 중심으로 인스턴스 정보를 추출하고,
# 해당 이벤트 내에서 생성 시점의 태그 정보를 가져오는 것을 목표로 합니다.
# CreateTags 이벤트를 별도로 조회하여 병합하는 것은 시작 시점의 태그를 특정하기 어렵고 복잡성을 증가시킬 수 있습니다.

def get_cloudtrail_events_with_retry(client: boto3.client, event_name: str, start_time: datetime, end_time: datetime, max_retries: int = 3, retry_delay_seconds: int = 10) -> list:
    """
    지정된 이벤트 이름에 대해 CloudTrail 이벤트를 가져옵니다.
    API 호출 제한(Rate Exceeded, ThrottlingException) 발생 시 재시도 로직을 포함합니다.
    """
    print(f"'{event_name}' 이벤트 조회를 시작합니다...")
    events_collected = []
    next_token = None
    page_num = 0
    
    while True:
        page_num += 1
        # print(f"  '{event_name}' 이벤트의 페이지 {page_num}을(를) 조회 중입니다...")
        params = {
            'LookupAttributes': [{'AttributeKey': 'EventName', 'AttributeValue': event_name}],
            'StartTime': start_time,
            'EndTime': end_time,
            'MaxResults': 50  # API 페이지 당 최대 결과 수
        }
        if next_token:
            params['NextToken'] = next_token

        current_retry = 0
        while current_retry <= max_retries:
            try:
                response = client.lookup_events(**params)
                fetched_on_page = response.get('Events', [])
                # print(f"  페이지 {page_num}에서 {len(fetched_on_page)}개의 '{event_name}' 이벤트를 가져왔습니다.")
                events_collected.extend(fetched_on_page)
                next_token = response.get('NextToken')
                break # 성공 시 내부 루프 탈출
            except client.exceptions.ClientError as e:
                error_code = e.response.get("Error", {}).get("Code")
                if error_code == "ThrottlingException" or "Rate exceeded" in str(e):
                    if current_retry < max_retries:
                        current_retry += 1
                        print(f"경고: '{event_name}' 이벤트 조회 중 API 호출 제한 발생. {retry_delay_seconds}초 후 재시도합니다... (시도 {current_retry}/{max_retries})")
                        time.sleep(retry_delay_seconds)
                    else:
                        print(f"오류: '{event_name}' 이벤트 조회 중 API 호출 제한이 {max_retries}회 초과되었습니다. 해당 이벤트의 추가 페이지 조회를 중단합니다.")
                        next_token = None # 더 이상 진행하지 않도록 next_token 제거
                        break # 내부 루프 탈출
                else:
                    print(f"오류: '{event_name}' 이벤트 조회 중 Boto3 ClientError 발생: {e}")
                    next_token = None 
                    break 
            except Exception as e:
                print(f"오류: '{event_name}' 이벤트 조회 중 예기치 않은 오류 발생: {e}")
                next_token = None 
                break 
        
        if not next_token:
            # print(f"  '{event_name}' 이벤트에 대한 더 이상 조회할 페이지가 없습니다.")
            break
            
    print(f"'{event_name}' 이벤트 조회 완료. 총 {len(events_collected)}개의 이벤트를 수집했습니다.")
    return events_collected

def format_tags_from_list(tags_list: list) -> str:
    """CloudTrail 이벤트 내의 태그 리스트를 'key1=value1;key2=value2' 형식의 문자열로 변환합니다."""
    if not tags_list:
        return ""
    
    formatted_tags = []
    for tag_item in tags_list:
        # CloudTrail 이벤트의 태그 구조는 {'key': 'KeyName', 'value': 'ValueName'} 또는 {'Key': ..., 'Value': ...} 일 수 있음
        key = tag_item.get('key', tag_item.get('Key'))
        value = tag_item.get('value', tag_item.get('Value'))
        if key is not None and value is not None: # 키와 값이 모두 존재해야 유효한 태그로 간주
             formatted_tags.append(f"{key}={value}")
    return ";".join(formatted_tags)


try:
    # AWS 인증 정보를 환경 변수, 공유 자격 증명 파일 또는 IAM 역할을 통해 Boto3가 자동으로 찾도록 합니다.
    # 특정 프로파일 사용 시: session = boto3.Session(profile_name="your-profile-name")
    # client = session.client('cloudtrail', region_name=REGION_NAME)
    cloudtrail_client = boto3.client('cloudtrail', region_name=REGION_NAME)
except Exception as e:
    print(f"오류: Boto3 CloudTrail 클라이언트를 초기화하는 중 문제가 발생했습니다: {e}")

# 조회할 시간 범위 설정 (UTC 기준)
utc_now = datetime.now(timezone.utc)
end_time_utc = utc_now
start_time_utc = utc_now - timedelta(days=LOOKBACK_DAYS)

print(f"{REGION_NAME} 리전에서 {start_time_utc.isoformat()} 부터 {end_time_utc.isoformat()} 까지의 CloudTrail 이벤트를 조회합니다.")

# 1. RunInstances 이벤트 조회
run_instances_events = get_cloudtrail_events_with_retry(
    cloudtrail_client, 
    EVENT_NAME_FOR_INSTANCE_CREATION, 
    start_time_utc, 
    end_time_utc
)

# 인스턴스ID 기준 병합을 위한 딕셔너리
instance_dict = {}

for event_log in run_instances_events:
    event_id = event_log.get('EventId', 'N/A')
    event_time_iso = event_log['EventTime'].isoformat() 
    cloud_trail_event_blob_str = event_log.get('CloudTrailEvent')
    if not cloud_trail_event_blob_str:
        print(f"경고: Event {event_id}에 CloudTrailEvent가 없습니다. 이벤트 로그를 무시합니다.")
        continue
    try:
        cloud_trail_event_blob = json.loads(cloud_trail_event_blob_str)
        event_name = cloud_trail_event_blob.get('eventName', 'N/A') 
        if event_name != EVENT_NAME_FOR_INSTANCE_CREATION:
            print(f"경고: Event {event_id}의 이벤트 이름이 '{EVENT_NAME_FOR_INSTANCE_CREATION}'가 아닙니다. 이벤트 로그를 무시합니다.")
            continue
        response_elements = cloud_trail_event_blob.get('responseElements')
        if not isinstance(response_elements, dict):
            response_elements = {}
        instances_set = response_elements.get('instancesSet')
        if not isinstance(instances_set, dict):
            instances_set = {}
        items = instances_set.get('items')
        if not isinstance(items, list) or not items or not isinstance(items[0], dict):
            items = [{}]
        instance_item = items[0]
        instance_id = instance_item.get('instanceId', 'N/A')
        instance_type = instance_item.get('instanceType', 'N/A')
        tagset = instance_item.get('tagSet')
        if not isinstance(tagset, dict):
            tagset = {}
        tags = format_tags_from_list(tagset.get('items', []))
        placement = instance_item.get('placement')
        if not isinstance(placement, dict):
            placement = {}
        az = placement.get('availabilityZone', '')
        instance_dict[instance_id] = {
            'event_id': event_id,
            'event_time': event_time_iso,
            'instance_id': instance_id,
            'instance_type': instance_type,
            'AZ': az,
            'tags': tags
        }
    except Exception as e:
        print(f"오류: Event {event_id}의 파싱 중 예기치 않은 오류 발생: {e}")
        continue

# 2. CreateTags 이벤트 조회 및 병합
create_tags_events = get_cloudtrail_events_with_retry(
    cloudtrail_client,
    'CreateTags',
    start_time_utc,
    end_time_utc
)

for event_log in create_tags_events:
    event_id = event_log.get('EventId', 'N/A')
    event_time_iso = event_log['EventTime'].isoformat()
    cloud_trail_event_blob_str = event_log.get('CloudTrailEvent')
    if not cloud_trail_event_blob_str:
        print(f"경고: Event {event_id}에 CloudTrailEvent가 없습니다. 이벤트 로그를 무시합니다.")
        continue
    try:
        cloud_trail_event_blob = json.loads(cloud_trail_event_blob_str)
        event_name = cloud_trail_event_blob.get('eventName', 'N/A')
        if event_name != 'CreateTags':
            print(f"경고: Event {event_id}의 이벤트 이름이 'CreateTags'가 아닙니다. 이벤트 로그를 무시합니다.")
            continue
        request_parameters = cloud_trail_event_blob.get('requestParameters')
        if not isinstance(request_parameters, dict):
            request_parameters = {}
        resources_set = request_parameters.get('resourcesSet')
        if not isinstance(resources_set, dict):
            resources_set = {}
        items = resources_set.get('items')
        if not isinstance(items, list) or not items or not isinstance(items[0], dict):
            items = [{}]
        resource_item = items[0]
        instance_id = resource_item.get('resourceId', 'N/A')
        tags_set = request_parameters.get('tagsSet')
        if not isinstance(tags_set, dict):
            tags_set = {}
        tags = format_tags_from_list(tags_set.get('items', []))
        if instance_id in instance_dict:
            run_tags = instance_dict[instance_id]['tags']
            merged_tags = run_tags
            if run_tags and tags:
                def tagstr_to_dict(tagstr):
                    d = {}
                    for t in tagstr.split(';'):
                        if '=' in t:
                            k, v = t.split('=', 1)
                            d[k] = v
                    return d
                merged = tagstr_to_dict(run_tags)
                merged.update(tagstr_to_dict(tags))
                merged_tags = ';'.join([f"{k}={v}" for k, v in merged.items()])
            elif tags:
                merged_tags = tags
            instance_dict[instance_id]['tags'] = merged_tags
        else:
            instance_dict[instance_id] = {
                'event_id': event_id,
                'event_time': event_time_iso,
                'instance_id': instance_id,
                'instance_type': '',
                'AZ': '',
                'tags': tags
            }
    except Exception as e:
        print(f"오류: Event {event_id}의 파싱 중 예기치 않은 오류 발생: {e}")
        continue

# 3. 결과 정렬 및 출력
# 인스턴스 ID를 기준으로 정렬
extracted_instance_data = list(instance_dict.values())
extracted_instance_data.sort(key=lambda x: x['instance_id'])

# 4. 필터링: 태그가 있고 특정 태그 값을 가진 인스턴스만 선택
filtered_instance_data = []
for instance in extracted_instance_data:
    if 'tags' in instance and instance['tags']:
        if 'eks:eks-cluster-name=kubecaps-d2-k8s-cluster' in instance['tags']:
            filtered_instance_data.append(instance)
print(f"정보: 총 {len(extracted_instance_data)}개 인스턴스 중 {len(filtered_instance_data)}개가 필터링 조건을 만족합니다.")
extracted_instance_data = filtered_instance_data

# CSV 파일 출력
if not extracted_instance_data:
    print("오류: 추출된 인스턴스 데이터가 없습니다. CSV 파일을 생성하지 않습니다.")
    

# 5. 태그 key를 열 단위로 확장하기 위해 모든 태그 key 수집
def tagstr_to_dict(tagstr):
    d = {}
    if tagstr:
        for t in tagstr.split(';'):
            if '=' in t:
                k, v = t.split('=', 1)
                d[k] = v
    return d

tag_keys = set()
for inst in extracted_instance_data:
    tag_keys.update(tagstr_to_dict(inst.get('tags', '')).keys())

base_fields = ['event_id', 'event_time', 'instance_id', 'instance_type', 'AZ']
fieldnames = base_fields + sorted(tag_keys)

# DataFrame 적재
rows = []
for inst in extracted_instance_data:
    row = {field: inst.get(field, '') for field in base_fields}
    tag_dict = tagstr_to_dict(inst.get('tags', ''))
    for k in tag_keys:
        row[k] = tag_dict.get(k, '')
    rows.append(row)
df = pd.DataFrame(rows, columns=fieldnames)

us-west-2 리전에서 2025-05-21T16:10:12.214533+00:00 부터 2025-05-22T16:10:12.214533+00:00 까지의 CloudTrail 이벤트를 조회합니다.
'RunInstances' 이벤트 조회를 시작합니다...
'RunInstances' 이벤트 조회 완료. 총 4494개의 이벤트를 수집했습니다.
'CreateTags' 이벤트 조회를 시작합니다...
'CreateTags' 이벤트 조회 완료. 총 1936개의 이벤트를 수집했습니다.
정보: 총 3273개 인스턴스 중 565개가 필터링 조건을 만족합니다.


In [16]:
run_instances_events[18]

{'EventId': '6922d78e-6d56-4bac-83de-9770a4b2cea3',
 'EventName': 'RunInstances',
 'ReadOnly': 'false',
 'EventTime': datetime.datetime(2025, 5, 23, 0, 50, 16, tzinfo=tzlocal()),
 'EventSource': 'ec2.amazonaws.com',
 'Username': 'InstanceLaunch',
 'Resources': [{'ResourceType': 'AWS::EC2::VPC',
   'ResourceName': 'vpc-0b4df2e6b97d58b9a'},
  {'ResourceType': 'AWS::EC2::Ami', 'ResourceName': 'ami-061211670aadd4f55'},
  {'ResourceType': 'AWS::EC2::NetworkInterface',
   'ResourceName': 'eni-0dfe1986e3f998b7a'},
  {'ResourceType': 'AWS::EC2::Instance',
   'ResourceName': 'i-0a332f8f3a595d1f7'},
  {'ResourceType': 'AWS::EC2::SecurityGroup',
   'ResourceName': 'terraform-2025052009414248650000000d'},
  {'ResourceType': 'AWS::EC2::SecurityGroup',
   'ResourceName': 'sg-0c3c6d877de889131'},
  {'ResourceType': 'AWS::EC2::Subnet',
   'ResourceName': 'subnet-0089fc9a6faac1aa0'}],
 'CloudTrailEvent': '{"eventVersion":"1.10","userIdentity":{"type":"AssumedRole","principalId":"AROAJYWRY7ZH2NLLUX3IS:I

In [9]:
import numpy as np
df['kubecaps-info'] = df['kubecaps-info'].replace('', np.nan)
df = df.dropna(subset=['kubecaps-info'])

In [11]:
# karpenter.k8s.aws/ec2nodeclass로 그룹화
grouped = df.groupby('karpenter.k8s.aws/ec2nodeclass')

result_rows = []

for nodeclass, group in grouped:
    # instance_type, AZ별로 몇 개의 인스턴스가 있는지 세기
    nodepool_config = []
    for (instance_type, az), subgrp in group.groupby(['instance_type', 'AZ']):
        config = {
            'instance_type': instance_type,
            'availability_zone': az,
            'num_instances': len(subgrp)
        }
        nodepool_config.append(config)
    
    # 가장 이른 event_time 구하기
    first_time = group['event_time'].min()
    
    first_row = group.iloc[0]
    result_row = {
        'pods': int(first_row['kubecaps-parallelism']),
        'cpu': int(first_row['kubecaps-cpu']),
        'memory': int(first_row['kubecaps-memgb']),
        'nodepool_config': nodepool_config,
        'first_time': first_time
    }
    result_rows.append(result_row)

result_df = pd.DataFrame(result_rows)
result_df.sort_values(by=['pods', 'cpu', 'memory'], ascending=False, inplace=True)
result_df.to_csv('result.csv', index=False, encoding='utf-8')
print(f"완료: 총 {df.shape[0]}개의 인스턴스 정보가 {OUTPUT_CSV_FILE}에 저장되었습니다.")

완료: 총 934개의 인스턴스 정보가 ec2_instance_details_us-west-2.csv에 저장되었습니다.


In [12]:
temp_df = result_df.copy()
temp_df['first_time'] = pd.to_datetime(temp_df['first_time'], errors='coerce')

start_time = pd.to_datetime("2025-05-21 17:00").tz_localize("Asia/Seoul")
end_time = pd.to_datetime("2025-05-21 19:00").tz_localize("Asia/Seoul")

filtered_df = temp_df[
    (temp_df["first_time"] > start_time) & (temp_df["first_time"] < end_time)
]


In [13]:
real_df = filtered_df.copy()
real_df.sort_values(by=['pods', 'cpu', 'memory'], inplace=True)

In [14]:
real_df.to_csv('result_3.csv', index=False, encoding='utf-8')

In [14]:
import boto3
import json
import os

session = boto3.Session(profile_name='spotrank')
def fetch_and_cache_az_mapping(region):
    ec2 = session.client('ec2', region_name=region)
    response = ec2.describe_availability_zones(AllAvailabilityZones=True)
    print(response)
    mapping = {az['ZoneName']: az['ZoneId'] for az in response['AvailabilityZones']}
    print(f"[Info] AZ Mapping for {region}: {mapping}")
    with open(f'{region}_az_mapping.json', 'w') as f:
        json.dump(mapping, f)
    return mapping


def load_az_mapping(region):
   fetch_and_cache_az_mapping(region)
    
def azname_to_azid(azname, region):
    mapping = load_az_mapping(region)
    return mapping[azname]

In [15]:
azname_to_azid("us-west-2c", "us-west-2")

{'AvailabilityZones': [{'OptInStatus': 'opt-in-not-required', 'Messages': [], 'RegionName': 'us-west-2', 'ZoneName': 'us-west-2a', 'ZoneId': 'usw2-az1', 'GroupName': 'us-west-2-zg-1', 'NetworkBorderGroup': 'us-west-2', 'ZoneType': 'availability-zone', 'GroupLongName': 'US West (Oregon) 1', 'State': 'available'}, {'OptInStatus': 'opt-in-not-required', 'Messages': [], 'RegionName': 'us-west-2', 'ZoneName': 'us-west-2b', 'ZoneId': 'usw2-az2', 'GroupName': 'us-west-2-zg-1', 'NetworkBorderGroup': 'us-west-2', 'ZoneType': 'availability-zone', 'GroupLongName': 'US West (Oregon) 1', 'State': 'available'}, {'OptInStatus': 'opt-in-not-required', 'Messages': [], 'RegionName': 'us-west-2', 'ZoneName': 'us-west-2c', 'ZoneId': 'usw2-az3', 'GroupName': 'us-west-2-zg-1', 'NetworkBorderGroup': 'us-west-2', 'ZoneType': 'availability-zone', 'GroupLongName': 'US West (Oregon) 1', 'State': 'available'}, {'OptInStatus': 'opt-in-not-required', 'Messages': [], 'RegionName': 'us-west-2', 'ZoneName': 'us-west-2

TypeError: 'NoneType' object is not subscriptable